# Group 2 — Age & Oxygen Saturation: Model Training Pipeline
**Week 5-6 deliverable: AI Model Evaluation & Research, Biomedical Data Analytics**

This builds on `group2_age_spo2_pipeline.ipynb` (data checks + simple train/test baselines) and
turns it into an actual model-training pipeline: proper 5-fold cross-validation, multiple model
types, hyperparameter selection, and a final held-out test evaluation — the structure needed to
report real results rather than exploratory numbers.

**Before running:**
1. Set `DATA_DIR` below.
2. This pipeline needs most/all of the local `ecg/` and `ppg/` folders downloaded (not just a
   handful of files) — the feature-based models will show `n/a` otherwise, exactly like last week.
3. Needs `scipy` and `scikit-learn` (`pip install scipy scikit-learn --break-system-packages`).


In [15]:
import json
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.signal import find_peaks

from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from numpy.random import default_rng

warnings.filterwarnings("ignore")

# ---- CONFIG ----
DATA_DIR = Path(r"C:\Users\User\mcd_rppg")   # e.g. Path(r"C:\Users\<you>\mcd_rppg")
DB_CSV = DATA_DIR / "db.csv"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 42
N_FOLDS = 5
TEST_FRAC = 0.2

assert DB_CSV.exists(), f"Could not find {DB_CSV} — update DATA_DIR above."
print("Using data dir:", DATA_DIR.resolve())


Using data dir: C:\Users\User\mcd_rppg


## 1. Load data, file readers, and feature extraction
Same readers and feature functions as the earlier notebook — included here so this file runs
standalone.


In [16]:
df = pd.read_csv(DB_CSV)
subj = df.drop_duplicates(["patient_id", "step"]).reset_index(drop=True)
print("Subject-state rows:", len(subj))


def load_pw(path):
    vals, times = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            vals.append(float(parts[0]))
            times.append(parts[1])
    return np.array(vals, dtype=float), times


def ppg_sampling_rate(times):
    fmt = "%Y-%m-%d %H:%M:%S.%f"
    t0 = datetime.datetime.strptime(times[0], fmt)
    t1 = datetime.datetime.strptime(times[-1], fmt)
    duration = (t1 - t0).total_seconds()
    return (len(times) / duration if duration > 0 else None), duration


def hrv_features(signal, fs, min_bpm=40, max_bpm=180):
    if fs is None:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi >= 60/max_bpm) & (ibi <= 60/min_bpm)]
    if len(ibi) < 3:
        return None
    hr = 60 / ibi
    nn_diff = np.diff(ibi) * 1000
    pnn50 = np.mean(np.abs(nn_diff) > 50) * 100 if len(nn_diff) else np.nan
    return {
        "mean_hr": hr.mean(), "hr_std": hr.std(), "hr_range": hr.max() - hr.min(),
        "mean_ibi": ibi.mean(), "sdnn": ibi.std() * 1000,
        "rmssd": np.sqrt(np.mean(nn_diff**2)) if len(nn_diff) else np.nan,
        "pnn50": pnn50, "n_beats": len(peaks),
    }


def sdppg_features(signal, fs, min_bpm=40, max_bpm=180):
    '''Simplified/heuristic SDPPG a/b-wave features — not a validated fiducial detector.'''
    if fs is None or len(signal) < 10:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    d2 = np.diff(signal, n=2)
    a_vals, b_vals = [], []
    for i in range(len(peaks) - 1):
        seg = d2[peaks[i]:peaks[i + 1]]
        if len(seg) < 5:
            continue
        half = seg[: max(len(seg) // 2, 3)]
        a, b = half.max(), half.min()
        if a > 0:
            a_vals.append(a)
            b_vals.append(b)
    if len(a_vals) < 3:
        return None
    a_vals, b_vals = np.array(a_vals), np.array(b_vals)
    return {"sdppg_a_mean": a_vals.mean(), "sdppg_b_mean": b_vals.mean(),
            "sdppg_ba_ratio": (b_vals / a_vals).mean(), "sdppg_n_beats": len(a_vals)}


Subject-state rows: 1200


In [17]:
rows = []
for _, r in subj.iterrows():
    stem = f"{r.patient_id}_{r.step}"
    pw_path = DATA_DIR / r.ppg
    row = {"subject_state": stem, "patient_id": r.patient_id, "step": r.step,
           "age": r.age, "sex": r.sex, "bmi": r.bmi, "saturation": r.saturation,
           "rigidity": r.get("rigidity", np.nan)}
    if pw_path.exists():
        try:
            vals, times = load_pw(pw_path)
            fs, dur = ppg_sampling_rate(times)
            hrv = hrv_features(vals, fs)
            sdp = sdppg_features(vals, fs)
            if hrv:
                row.update(hrv)
            if sdp:
                row.update(sdp)
            row["ppg_found"] = True
        except Exception as e:
            print(f"  skipping {pw_path.name}: {e}")
            row["ppg_found"] = False
            row["ppg_error"] = str(e)
    else:
        row["ppg_found"] = False
    rows.append(row)

features = pd.DataFrame(rows)
features["sex_num"] = (features["sex"] == "M").astype(int)

HRV_COLS = ["mean_hr", "hr_std", "hr_range", "mean_ibi", "sdnn", "rmssd", "pnn50"]
SDPPG_COLS = ["sdppg_a_mean", "sdppg_b_mean", "sdppg_ba_ratio"]
FEATURE_COLS = HRV_COLS + SDPPG_COLS

n_usable = features.dropna(subset=FEATURE_COLS).shape[0]
print(f"PPG files found locally: {features.ppg_found.sum()} / {len(features)}")
print(f"Rows with a complete feature set: {n_usable} / {len(features)}")
if n_usable < 100:
    print()
    print("WARNING: fewer than 100 usable rows. Model results below will be unreliable until")
    print("more of the local ppg/ folder is downloaded. The pipeline will still run.")


  skipping 8706_after.PW: time data '2023-10-' does not match format '%Y-%m-%d %H:%M:%S.%f'
PPG files found locally: 1199 / 1200
Rows with a complete feature set: 1199 / 1200


## 2. Subject-level train / test split (held out until the very end)
Same method as before: split by `patient_id`, not by row, with zero overlap. The test set is
**only touched once**, at the end, after model selection is finished on the training set.


In [18]:
def subject_level_split(subject_ids, test_frac=TEST_FRAC, seed=SEED):
    ids = np.array(sorted(set(subject_ids)))
    rng = default_rng(seed)
    rng.shuffle(ids)
    n_test = int(len(ids) * test_frac)
    return set(ids[n_test:]), set(ids[:n_test])  # train_ids, test_ids

train_ids, test_ids = subject_level_split(features.patient_id.unique())
assert train_ids.isdisjoint(test_ids)
print(f"Train subjects: {len(train_ids)}  Test subjects: {len(test_ids)}  Overlap: 0 (verified)")


Train subjects: 480  Test subjects: 120  Overlap: 0 (verified)


## 3. Cross-validated model comparison (training set only)
For each target, compare several model types using 5-fold **group** cross-validation (grouped by
`patient_id`, so no subject appears in both the CV-train and CV-validation fold). This is how the
best model is chosen — the held-out test set in Section 4 is not used here.


In [19]:
def bootstrap_mae_ci(y_true, y_pred, n_boot=1000, seed=0):
    rng = default_rng(seed)
    n = len(y_true)
    errs = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    boot_means = [errs[rng.integers(0, n, n)].mean() for _ in range(n_boot)]
    return np.mean(errs), np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)


def cv_compare_models(data, target_col, demo_cols, feature_cols, group_col="patient_id",
                       n_folds=N_FOLDS, seed=SEED):
    '''Returns a DataFrame of per-fold MAE for each candidate model.'''
    data = data.dropna(subset=[target_col]).copy()
    groups = data[group_col].values

    gkf = GroupKFold(n_splits=n_folds)
    records = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(data, groups=groups)):
        tr, va = data.iloc[tr_idx], data.iloc[va_idx]

        # 1. mean predictor
        pred = np.full(len(va), tr[target_col].mean())
        records.append({"fold": fold, "model": "mean_predictor",
                         "mae": mean_absolute_error(va[target_col], pred), "n_val": len(va)})

        # 2. demographics-only (Ridge)
        d_tr = tr.dropna(subset=demo_cols + [target_col])
        d_va = va.dropna(subset=demo_cols + [target_col])
        if len(d_tr) > 10 and len(d_va) > 3:
            scaler = StandardScaler().fit(d_tr[demo_cols])
            model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]).fit(
                scaler.transform(d_tr[demo_cols]), d_tr[target_col])
            pred = model.predict(scaler.transform(d_va[demo_cols]))
            records.append({"fold": fold, "model": "demographics_ridge",
                             "mae": mean_absolute_error(d_va[target_col], pred), "n_val": len(d_va)})

        # 3. feature-based Ridge
        f_tr = tr.dropna(subset=feature_cols + [target_col])
        f_va = va.dropna(subset=feature_cols + [target_col])
        if len(f_tr) > 10 and len(f_va) > 3:
            scaler = StandardScaler().fit(f_tr[feature_cols])
            model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]).fit(
                scaler.transform(f_tr[feature_cols]), f_tr[target_col])
            pred = model.predict(scaler.transform(f_va[feature_cols]))
            records.append({"fold": fold, "model": "feature_ridge",
                             "mae": mean_absolute_error(f_va[target_col], pred), "n_val": len(f_va)})

        # 4. feature-based Random Forest (captures non-linearity, different failure mode to Ridge)
        if len(f_tr) > 20 and len(f_va) > 3:
            rf = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=seed,
                                        min_samples_leaf=5)
            rf.fit(f_tr[feature_cols], f_tr[target_col])
            pred = rf.predict(f_va[feature_cols])
            records.append({"fold": fold, "model": "feature_random_forest",
                             "mae": mean_absolute_error(f_va[target_col], pred), "n_val": len(f_va)})

    return pd.DataFrame(records)


def summarise_cv(cv_df):
    summary = cv_df.groupby("model")["mae"].agg(["mean", "std", "count"]).reset_index()
    summary = summary.sort_values("mean")
    summary.columns = ["model", "cv_mae_mean", "cv_mae_std", "n_folds_run"]
    return summary


In [20]:
train_data = features[features.patient_id.isin(train_ids)]

AGE_DEMO_COLS = ["sex_num", "bmi"]
SAT_DEMO_COLS = ["sex_num", "age", "bmi"]

print("=" * 60)
print("AGE — cross-validated model comparison (training set)")
print("=" * 60)
age_cv = cv_compare_models(train_data, "age", AGE_DEMO_COLS, FEATURE_COLS)
age_summary = summarise_cv(age_cv)
print(age_summary.to_string(index=False))


AGE — cross-validated model comparison (training set)
                model  cv_mae_mean  cv_mae_std  n_folds_run
feature_random_forest     3.384592    0.413300            5
   demographics_ridge     5.047876    0.630040            5
       mean_predictor     5.093804    0.556480            5
        feature_ridge     5.260153    0.318305            5


In [21]:
print("=" * 60)
print("SATURATION — cross-validated model comparison (training set)")
print("=" * 60)
sat_cv = cv_compare_models(train_data, "saturation", SAT_DEMO_COLS, HRV_COLS)
sat_summary = summarise_cv(sat_cv)
print(sat_summary.to_string(index=False))


SATURATION — cross-validated model comparison (training set)
                model  cv_mae_mean  cv_mae_std  n_folds_run
   demographics_ridge     0.852681    0.057050            5
       mean_predictor     0.857010    0.051513            5
        feature_ridge     0.875831    0.052277            5
feature_random_forest     0.891189    0.050444            5


## 4. Final held-out test evaluation
Refit the **best model per target** (lowest CV MAE from Section 3) on the *full* training set,
then evaluate once on the held-out test subjects. This number — not the CV number — is what goes
in the report, since the test set was never used for model selection.


In [22]:
def fit_and_test(train_data, test_data, target_col, feature_cols, model_type):
    tr = train_data.dropna(subset=feature_cols + [target_col])
    te = test_data.dropna(subset=feature_cols + [target_col])
    if len(tr) < 10 or len(te) < 3:
        print(f"  Not enough data to evaluate on the held-out test set "
              f"(train={len(tr)}, test={len(te)}). Download more PPG files and re-run.")
        return None

    if model_type == "mean_predictor":
        pred = np.full(len(te), tr[target_col].mean())
    elif model_type in ("demographics_ridge", "feature_ridge"):
        scaler = StandardScaler().fit(tr[feature_cols])
        model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]).fit(
            scaler.transform(tr[feature_cols]), tr[target_col])
        pred = model.predict(scaler.transform(te[feature_cols]))
    elif model_type == "feature_random_forest":
        model = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=SEED,
                                       min_samples_leaf=5)
        model.fit(tr[feature_cols], tr[target_col])
        pred = model.predict(te[feature_cols])
    else:
        raise ValueError(model_type)

    mae, lo, hi = bootstrap_mae_ci(te[target_col], pred)
    print(f"  Held-out test MAE ({model_type}): {mae:.2f} [{lo:.2f}, {hi:.2f}]  (n={len(te)})")
    return {"target": target_col, "model": model_type, "test_mae": mae, "ci_lo": lo, "ci_hi": hi,
            "n_test": len(te)}


test_data = features[features.patient_id.isin(test_ids)]

print("AGE — held-out test evaluation")
best_age_model = age_summary.iloc[0]["model"]
age_feat_cols = FEATURE_COLS if "feature" in best_age_model else AGE_DEMO_COLS
age_test_result = fit_and_test(train_data, test_data, "age", age_feat_cols, best_age_model)

print()
print("SATURATION — held-out test evaluation")
best_sat_model = sat_summary.iloc[0]["model"]
sat_feat_cols = HRV_COLS if "feature" in best_sat_model else SAT_DEMO_COLS
sat_test_result = fit_and_test(train_data, test_data, "saturation", sat_feat_cols, best_sat_model)


AGE — held-out test evaluation
  Held-out test MAE (feature_random_forest): 3.87 [3.03, 4.80]  (n=240)

SATURATION — held-out test evaluation
  Held-out test MAE (demographics_ridge): 0.91 [0.81, 1.03]  (n=240)


## 5. Saturation tail-subset evaluation
As noted in last week's report: saturation's narrow range means the overall MAE can look good by
default. Report the best model's error on the low-saturation (below 95%) subset separately.


In [23]:
tail_test = test_data[test_data["saturation"] < 95]
print(f"Low-saturation test rows (<95%): {len(tail_test)}")
if len(tail_test) >= 3 and sat_test_result is not None:
    tail_feat = tail_test.dropna(subset=sat_feat_cols + ["saturation"])
    if len(tail_feat) >= 3:
        tr = train_data.dropna(subset=sat_feat_cols + ["saturation"])
        if best_sat_model == "mean_predictor":
            pred = np.full(len(tail_feat), tr["saturation"].mean())
        else:
            scaler = StandardScaler().fit(tr[sat_feat_cols])
            model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]).fit(
                scaler.transform(tr[sat_feat_cols]), tr["saturation"])
            pred = model.predict(scaler.transform(tail_feat[sat_feat_cols]))
        mae, lo, hi = bootstrap_mae_ci(tail_feat["saturation"], pred)
        print(f"Tail-subset MAE ({best_sat_model}): {mae:.2f} [{lo:.2f}, {hi:.2f}]")
        print("Compare this against the overall test MAE above — expect it to be noticeably worse.")
    else:
        print("Not enough tail rows with full features yet.")
else:
    print("Not enough low-saturation test rows to evaluate separately yet.")


Low-saturation test rows (<95%): 8
Tail-subset MAE (demographics_ridge): 3.91 [3.22, 4.67]
Compare this against the overall test MAE above — expect it to be noticeably worse.


## 6. Save results for the report / presentation

In [24]:
results_rows = []
if age_test_result:
    results_rows.append(age_test_result)
if sat_test_result:
    results_rows.append(sat_test_result)

results_df = pd.DataFrame(results_rows)
out_path = RESULTS_DIR / "model_results_summary.csv"
results_df.to_csv(out_path, index=False)
age_summary.to_csv(RESULTS_DIR / "age_cv_comparison.csv", index=False)
sat_summary.to_csv(RESULTS_DIR / "saturation_cv_comparison.csv", index=False)

print(f"Saved: {out_path}")
print(f"Saved: {RESULTS_DIR / 'age_cv_comparison.csv'}")
print(f"Saved: {RESULTS_DIR / 'saturation_cv_comparison.csv'}")
print()
print(results_df.to_string(index=False) if len(results_df) else "No results to save yet — "
      "run after downloading more PPG files.")


Saved: C:\Users\User\mcd_rppg\results\model_results_summary.csv
Saved: C:\Users\User\mcd_rppg\results\age_cv_comparison.csv
Saved: C:\Users\User\mcd_rppg\results\saturation_cv_comparison.csv

    target                 model  test_mae    ci_lo    ci_hi  n_test
       age feature_random_forest  3.872838 3.033302 4.798829     240
saturation    demographics_ridge  0.912673 0.806239 1.032657     240


## 7. Summary checklist for the report

In [25]:
print("=" * 60)
print("SUMMARY CHECKLIST")
print("=" * 60)
checklist = [
    "[ ] Confirmed enough local PPG/ECG data downloaded (100+ usable rows minimum)",
    "[ ] 5-fold group cross-validation run for age, comparing 4 model types",
    "[ ] 5-fold group cross-validation run for saturation, comparing 4 model types",
    "[ ] Best model per target selected by CV MAE (not by peeking at the test set)",
    "[ ] Held-out test MAE computed once, for the selected model only",
    "[ ] Age test result compared against published 9-10 year PPG-only MAE range",
    "[ ] Saturation tail-subset (<95%) MAE reported separately from overall MAE",
    "[ ] Results saved to results/ for the presentation",
    "[ ] Noted: SDPPG features are heuristic, not a validated fiducial detector",
    "[ ] Noted: no RGB colour-ratio features used for saturation (PW is single-channel)",
]
for item in checklist:
    print(item)


SUMMARY CHECKLIST
[ ] Confirmed enough local PPG/ECG data downloaded (100+ usable rows minimum)
[ ] 5-fold group cross-validation run for age, comparing 4 model types
[ ] 5-fold group cross-validation run for saturation, comparing 4 model types
[ ] Best model per target selected by CV MAE (not by peeking at the test set)
[ ] Held-out test MAE computed once, for the selected model only
[ ] Age test result compared against published 9-10 year PPG-only MAE range
[ ] Saturation tail-subset (<95%) MAE reported separately from overall MAE
[ ] Results saved to results/ for the presentation
[ ] Noted: SDPPG features are heuristic, not a validated fiducial detector
[ ] Noted: no RGB colour-ratio features used for saturation (PW is single-channel)


In [26]:
# Stratified check: is the age model actually good, or just good on the common young cluster?
test_age_data = test_data.dropna(subset=age_feat_cols + ["age"]).copy()
tr = train_data.dropna(subset=age_feat_cols + ["age"])

model = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=SEED, min_samples_leaf=5)
model.fit(tr[age_feat_cols], tr["age"])
test_age_data["pred"] = model.predict(test_age_data[age_feat_cols])
test_age_data["abs_err"] = (test_age_data["pred"] - test_age_data["age"]).abs()

young = test_age_data[test_age_data["age"] < 25]
old = test_age_data[test_age_data["age"] >= 25]

print(f"Young (<25), n={len(young)}: MAE = {young['abs_err'].mean():.2f}")
print(f"Old (25+), n={len(old)}: MAE = {old['abs_err'].mean():.2f}")
print()
print("Feature importances:")
for name, imp in sorted(zip(age_feat_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:20s} {imp:.3f}")

Young (<25), n=210: MAE = 2.77
Old (25+), n=30: MAE = 11.60

Feature importances:
  hr_std               0.511
  sdppg_b_mean         0.108
  pnn50                0.106
  hr_range             0.104
  mean_hr              0.046
  sdppg_a_mean         0.043
  mean_ibi             0.039
  sdppg_ba_ratio       0.017
  rmssd                0.016
  sdnn                 0.009
